# Assignment 7: Student Habits Dataset


## Part 1 — Load the Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("class1_student_habits.csv")
df.head(10)

## Questions for Part 1

What is a Pandas DataFrame? A DataFrame is Pandas' main table type. It's a two-dimensional grid of rows and columns, similar to a spreadsheet or a database table. Each column has a name and a type, and you can select, filter, and summarize the data with the built in Pandas methods.

What does one row represent in this dataset? One row represents one student. All of the numbers and labels in that row belong to that one student.

What does one column represent? One column is one measured attribute for a given student. The column name is the variable, and the cells under it are that variable's values across students. Some cells may be empty, which just means we are missing some values.

## Part 2 — Inspect the Dataset

This part prints the number of rows and columns, the column names, each column's data type, and how many missing values each column has.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print()
print("Column names:")
print(list(df.columns))
print()
print("Data type of each column:")
print(df.dtypes)
print()
print("Missing values in each column:")
print(df.isna().sum())

## Questions for Part 2

Which columns are numerical? study_hours_per_week, sleep_hours_per_night, attendance_rate, practice_problems_per_week, commute_minutes, quiz_average, and exam_score. 

Which columns are categorical? prior_programming_experience and study_group. 

Which column is an identifier? student_id: Each value is unique (S001, S002, ...) and names the student rather than measuring a habit or score.

Which columns contain missing values? study_hours_per_week, sleep_hours_per_night, and prior_programming_experience.

## Part 3 — Descriptive Statistics

This part summarizes the numerical columns: count, mean, median, standard deviation, minimum, and maximum. Count is lower than 250 for study hours and sleep hours because those columns have missing values.

In [ ]:
numeric_cols = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes",
    "quiz_average",
    "exam_score",
]

df[numeric_cols].agg(["count", "mean", "median", "std", "min", "max"])

## Questions for Part 3

Three interesting observations:

1. Most students are doing average on tests. The typical quiz average and exam score both sit in the mid-70s, and the middle of the class is almost the same as the average. That means this is not a class split into a pile of A's and a pile of failing scores. A typical student is in the C+/B- range, with a smaller number at the top and a smaller number in the 50s.

2. Sleep is the most consistent habit. Students usually sleep about seven hours a night, and most people are within about an hour of that. The distribution is much tighter than study time or commute time. 

3. Commute time and practice problems both have a long tail. The typical commute is a bit over 20 minutes, and some students have no commute at all, but at least one student spends more than two hours getting to class. Time spent on practice is similar, as the typical student does about 16 problems a week, some do none, and one student did 78. Those outliers pull the average up a little compared with the middle of the class, so the mean of both of these catergories is not totally representative of the entire sample. 

## Part 4 — Filtering, Sorting, and Grouping

### A. Filtering

Students who study at least 10 hours per week and attend at least 90% of class.

In [ ]:
high_study_and_attendance = df[
    (df["study_hours_per_week"] >= 10) & (df["attendance_rate"] >= 90)
]

useful_cols = [
    "student_id",
    "study_hours_per_week",
    "attendance_rate",
    "practice_problems_per_week",
    "study_group",
    "quiz_average",
    "exam_score",
]

print("Number of students:", len(high_study_and_attendance))
high_study_and_attendance[useful_cols]

### B. Sorting

The 10 students with the highest exam scores.

In [ ]:
top10_exam = df.sort_values("exam_score", ascending=False).head(10)

top10_exam[
    [
        "student_id",
        "exam_score",
        "quiz_average",
        "study_hours_per_week",
        "attendance_rate",
        "practice_problems_per_week",
        "study_group",
        "prior_programming_experience",
    ]
]

### C. Grouping

Exam scores for students who do and do not participate in a study group.

In [ ]:
df.groupby("study_group")["exam_score"].agg(["count", "mean", "median"])

## Question for Part 4

What do you observe from the group comparison?

The two groups are almost the same size, and their exam scores look almost the same. Students in a study group have a slightly higher average and a slightly higher middle score than students who are not in a group, but the gap is small. Inside each group, scores still range from the 50s to 100, so the overlap is large. The small difference does not mean joining a study group raises exam scores. Students who already do well, study more, or like working with others may be more likely to join a group, and other habits (sleep, practice, attendance, or prior experience) could also differ between the two groups. This table only describes the two categories in this class; it does not show that study-group participation caused the difference.

## Part 5 — Visualize the Data

### Visualization 1 — Histogram of exam scores

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(df["exam_score"], bins=10, edgecolor="black")
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam score")
plt.ylabel("Number of students")
plt.show()

Interpretation: Most exam scores pile up in the 70s and 80s. Relatively few students scored in the 50s, and relatively few scored in the 90s or got perfect scores. The shape is one hump around the middle of the class, which matches the earlier observation that a typical student is in the C+/B- range.

### Visualization 2 — Study hours vs exam score

In [ ]:
study_exam = df.dropna(subset=["study_hours_per_week", "exam_score"])

plt.figure()
plt.scatter(study_exam["study_hours_per_week"], study_exam["exam_score"], alpha=0.7)
plt.title("Study Hours per Week vs Exam Score")
plt.xlabel("Study hours per week")
plt.ylabel("Exam score")
plt.show()

Interpretation: For this relationship there is an upward trend. It shows that students who study more hours in a week often score higher on the exam. The scatter is still wide, so hours alone do not guarantee a high score. Some students study a lot and still land in the 60s, and some study fewer hours and still do well. The plot shows simply possible correlation, not causation.

### Visualization 3 — Quiz average vs exam score

Question: Do students who do better on quizzes also do better on the exam?

In [ ]:
plt.figure()
plt.scatter(df["quiz_average"], df["exam_score"], alpha=0.7)
plt.title("Quiz Average vs Exam Score")
plt.xlabel("Quiz average")
plt.ylabel("Exam score")
plt.show()

Interpretation: This shows that quiz average leads to exam score more strongly than study hours do. Students with stronger quiz averages generally also have stronger exam scores, and the points form a clearer upward curve. The match is not perfect, as some students do better on quizzes than on the exam, or vice versa.

## Part 6 — Investigate a Relationship

Chosen variable: practice_problems_per_week

My prediction: I expect practice problems per week to be strongly associated with exam score because students who complete more practice get extra chances to work problems like the ones on the exam.

This part creates a scatter plot and computes the Pearson correlation.

In [ ]:
plt.figure()
plt.scatter(df["practice_problems_per_week"], df["exam_score"], alpha=0.7)
plt.title("Practice Problems per Week vs Exam Score")
plt.xlabel("Practice problems per week")
plt.ylabel("Exam score")
plt.show()

r = df["practice_problems_per_week"].corr(df["exam_score"])
print("Correlation:", r)

The correlation is about 0.34, which is a moderate positive association. Students who complete more practice problems in a week tend to have higher exam scores in this dataset, but the relationship is not strong as I predicted. Many students with similar practice counts still have very different exam scores, and a few students with a lot of practice do not sit at the top of the exam. Correlation does not mean that assigning extra practice problems would raise a given student's exam score by a certain number of points. Students who already understand the material, or are maybe more motivated, may both practice more and score higher.

## Part 7 — Ask Your Own Data Question

Question: Is sleep time related to study hours?

Data needed reccomended by assitant: sleep_hours_per_night and study_hours_per_week. Drop rows where either value is missing, so each point is a student with both measurements.

Useful Pandas operations reccomended by assistant: dropna on those two variables mentioned, then corr to measure how they move together.

Visualization reccomended by assitant: a scatter plot of sleep hours vs study hours. If they were related, the points would slope up or down. If not, the points would look like a shapeless patch.

In [ ]:
sleep_study = df.dropna(subset=["sleep_hours_per_night", "study_hours_per_week"])

print("Number of students with both values:", len(sleep_study))
print("Correlation:", sleep_study["sleep_hours_per_night"].corr(sleep_study["study_hours_per_week"]))
print()
print(sleep_study[["sleep_hours_per_night", "study_hours_per_week"]].agg(["count", "mean", "median", "std"]))

plt.figure()
plt.scatter(sleep_study["sleep_hours_per_night"], sleep_study["study_hours_per_week"], alpha=0.7)
plt.title("Sleep Hours per Night vs Study Hours per Week")
plt.xlabel("Sleep hours per night")
plt.ylabel("Study hours per week")
plt.show()

Conclusion: Sleep time is not related to study hours in this dataset. After dropping students missing either value, 230 students remain, and the correlation comes to about 0. The scatter plot has no clear upward or downward slope, and students who sleep about 5 hours and students who sleep about 9 hours both study a wide range of weekly hours. A possible guess is that short sleepers stay up to study more, however these numbers do not support that either. The two habits look independent.

## Part 8 — AI Reflection

Situation: Building my own data question

The assistant suggested how to test whether sleep time is related to study hours. It said to keep only students who had both values, compute a correlation, and then make a scatter plot of sleep vs study hours. 

How I checked it: 
I ran the cells in Colab and looked at the printed numbers. For the study-group comparison, the table showed almost the same mean and median exam scores for Yes and No, so I kept the conclusion that the groups look similar and that this does not prove study groups cause higher scores. For sleep vs study hours, the correlation printed as about 0 and the scatter had no slope, which matched the assistant's reading.

Something I did not know before: I did not know I had to drop missing values in both columns first, or Pandas would be correlating a smaller, mismatched set of rows.